# Inferring cell-specific GRNs in heterogeneous cell populations

A heterogeneous tissue contains cells with different identities and developmental states. These cells may use different regulatory programs, even when they were measured in the same experiment. A single pooled network can hide these differences.

scCAFM therefore generates a separate **cell-specific gene regulatory network (GRN)** for every cell. In each network:

- `Gene1` is a transcription factor (TF);
- `Gene2` is a possible target gene;
- `score` is the predicted regulatory strength for that cell.

Here we use an E15.5 mouse-pancreas dataset containing eight annotated cell populations. The tutorial focuses only on generating and inspecting cell-specific GRNs. It does not perform evaluation, clustering, trajectory analysis, or other downstream tasks.


## 1. Set up the tutorial

The model files are read from `assets/`. The mouse-pancreas dataset is read from `tutorial_data/cell_specific_grns/`. All paths are relative to the repository.


In [1]:
from pathlib import Path
import warnings

from IPython.display import display
import pandas as pd
import scanpy as sc
import torch

warnings.filterwarnings(
    "ignore",
    message="Mismatch dtype between input and weight",
)

from sccafm import (
    GRNInferencer,
    ScPreprocessor,
    load_vocab_json,
    resolve_model_assets,
    write_cell_specific_grns_csv,
)


REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pyproject.toml").is_file():
    REPO_ROOT = REPO_ROOT.parent

MODEL_SOURCE = REPO_ROOT / "assets"
DATA_PATH = (
    REPO_ROOT
    / "tutorial_data"
    / "cell_specific_grns"
    / "mPancreas.h5ad"
)

if not DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Place the mouse-pancreas dataset at: {DATA_PATH}"
    )

assets = resolve_model_assets(MODEL_SOURCE)
token_dict = load_vocab_json(assets.vocab)
mouse_tfs = pd.read_csv(assets.mouse_tfs)["TF"].astype(str).tolist()
print("Tutorial files are ready.")


Tutorial files are ready.


## 2. Examine the mouse-pancreas data

This dataset is the E15.5 subset of a developing 10x mouse-pancreas experiment. It contains ductal cells, endocrine progenitors, and four endocrine cell types. The table is generated directly from the tutorial file.

The species is set explicitly to `"mouse"` in memory because scCAFM uses this field to select the appropriate TF catalogue. The file on disk is not modified.


In [2]:
raw_adata = sc.read_h5ad(DATA_PATH)
raw_shape = raw_adata.shape
raw_adata.obs["species"] = "mouse"

if "clusters" not in raw_adata.obs:
    raise KeyError("The tutorial dataset must contain adata.obs['clusters'].")

population_counts = (
    raw_adata.obs["clusters"]
    .value_counts(sort=False)
    .rename_axis("Cell population")
    .reset_index(name="Cells")
)

dataset_overview = pd.DataFrame(
    [
        {
            "Dataset": "mPancreas",
            "Species": "mouse",
            "Stage": "E15.5",
            "Cells": raw_adata.n_obs,
            "Measured genes": raw_adata.n_vars,
            "Cell populations": raw_adata.obs["clusters"].nunique(),
        }
    ]
)

display(
    dataset_overview.style.format(
        {
            "Cells": "{:,.0f}",
            "Measured genes": "{:,.0f}",
            "Cell populations": "{:,.0f}",
        }
    )
)
display(population_counts.style.format({"Cells": "{:,.0f}"}))


,Dataset,Species,Stage,Cells,Measured genes,Cell populations
0,mPancreas,mouse,E15.5,"3,696","27,998",8


,Cell population,Cells
0,Ductal,916
1,Ngn3 low EP,262
2,Ngn3 high EP,642
3,Pre-endocrine,592
4,Beta,591
5,Alpha,481
6,Delta,70
7,Epsilon,142


## 3. Prepare the expression data

We retain every available mouse TF so that it can act as a regulator. We then select 500 variable non-TF genes as additional targets. This smaller gene set keeps the tutorial fast while preserving regulatory coverage.

The preprocessing step filters low-quality cells and genes, normalizes each cell to a total count of 10000, removes common technical gene groups, and keeps only genes recognized by scCAFM. Preprocessing returns a new `AnnData` object, and the H5AD file on disk is not modified.


In [3]:
preprocessor = ScPreprocessor(
    min_genes=200,
    min_cells=3,
    max_pct_counts_mt=20.0,
    target_sum=10000,
    log1p=False,
    n_top_genes=500,
    hvg_flavor="seurat",
    subset_hvg=True,
    remove_mito_genes=True,
    remove_ribo_genes=True,
    remove_hb_genes=True,
    token_dict=token_dict,
    preserve_gene_names=mouse_tfs,
    hvg_exclude_gene_names=mouse_tfs,
    hvg_exclude_preserved_genes=True,
    sanitize_X=True,
    inplace=False,
)
adata = preprocessor(raw_adata)

mouse_tf_set = {name.strip().upper() for name in mouse_tfs}
retained_genes = {str(name).strip().upper() for name in adata.var_names}
n_retained_tfs = len(retained_genes & mouse_tf_set)
n_retained_non_tfs = adata.n_vars - n_retained_tfs

prepared_overview = pd.DataFrame(
    [
        {
            "Cells before": raw_shape[0],
            "Cells after": adata.n_obs,
            "Genes before": raw_shape[1],
            "TFs retained": n_retained_tfs,
            "Variable non-TFs": n_retained_non_tfs,
            "Total genes": adata.n_vars,
        }
    ]
)
display(
    prepared_overview.style.format(
        {
            "Cells before": "{:,.0f}",
            "Cells after": "{:,.0f}",
            "Genes before": "{:,.0f}",
            "TFs retained": "{:,.0f}",
            "Variable non-TFs": "{:,.0f}",
            "Total genes": "{:,.0f}",
        }
    )
)


,Cells before,Cells after,Genes before,TFs retained,Variable non-TFs,Total genes
0,"3,696","3,696","27,998","1,099",500,"1,599"


## 4. Generate cell-specific GRNs

The model is loaded once on one GPU. This tutorial uses `attention_backend="fa4"`; use `"fa2"` instead when FA4 is unavailable and the FA2 kernels are installed.

We retain edges with `score >= 0.1` independently in every cell. The threshold is applied after scCAFM reconstructs each raw cell-specific GRN. It keeps the resulting edge tables and exports much smaller than the complete networks.

A score threshold is a user-chosen cutoff, not a statistical significance level. Different analyses may require a different value. The optional export section also shows how to switch to a fixed top-k selection.

`CellSpecificGRNs` is lazy: it stores the prepared inputs and runs model inference when the networks are iterated or written. This avoids keeping every cell's complete GRN in memory at once.


In [4]:
if not torch.cuda.is_available():
    raise RuntimeError("This tutorial requires one CUDA GPU.")

inferencer = GRNInferencer.from_pretrained(
    MODEL_SOURCE,
    device="cuda:0",
    attention_backend="fa4",  # Use "fa2" if FA4 is not available.
    max_length=2048,
    species_key="species",
    disease_key="disease",
)

SCORE_THRESHOLD = 0.1
cell_grns = inferencer.infer_cell_specific(
    adata,
    batch_size=8,
    score_threshold=SCORE_THRESHOLD,
)

candidate_edges_per_cell = cell_grns.shape[1] * cell_grns.shape[2]
result_overview = pd.DataFrame(
    [
        {
            "Cells": cell_grns.shape[0],
            "Source TFs": cell_grns.shape[1],
            "Target genes": cell_grns.shape[2],
            "Candidate edges per cell": candidate_edges_per_cell,
            "Score threshold": SCORE_THRESHOLD,
        }
    ]
)

print("The thresholded cell-specific GRN collection is ready for iteration.")
display(
    result_overview.style.format(
        {
            "Cells": "{:,.0f}",
            "Source TFs": "{:,.0f}",
            "Target genes": "{:,.0f}",
            "Candidate edges per cell": "{:,.0f}",
            "Score threshold": "{:.1f}",
        }
    )
)


The thresholded cell-specific GRN collection is ready for iteration.


,Cells,Source TFs,Target genes,Candidate edges per cell,Score threshold
0,"3,696","1,099","1,599","1,757,301",0.1


## 5. Inspect representative cells

To keep the preview compact, we select one cell from each annotated population. The table shows the five strongest retained edges with `score >= 0.1` for every representative cell.


In [5]:
cell_type_order = list(adata.obs["clusters"].cat.categories)
representative_cell_ids = [
    str(adata.obs_names[adata.obs["clusters"] == cell_type][0])
    for cell_type in cell_type_order
]
representative_adata = adata[representative_cell_ids].copy()

representative_grns = inferencer.infer_cell_specific(
    representative_adata,
    batch_size=8,
    score_threshold=SCORE_THRESHOLD,
)
cell_type_by_id = (
    representative_adata.obs["clusters"].astype(str).to_dict()
)

preview_tables = []
for edge_table in representative_grns.iter_edge_tables():
    edge_table.insert(
        1,
        "Cell type",
        edge_table["cell_id"].map(cell_type_by_id),
    )
    preview_tables.append(edge_table.nlargest(5, "score"))

representative_edges = pd.concat(preview_tables, ignore_index=True)
representative_edges = representative_edges[
    ["Cell type", "cell_id", "Gene1", "Gene2", "score"]
]
display(representative_edges.style.format({"score": "{:.4f}"}))


,Cell type,cell_id,Gene1,Gene2,score
0,Ductal,AAACCTGAGCCTTGAT,Med13,Pcbp2,0.5152
1,Ductal,AAACCTGAGCCTTGAT,Srsf9,Pcbp2,0.5119
2,Ductal,AAACCTGAGCCTTGAT,Med13,Cited1,0.5056
3,Ductal,AAACCTGAGCCTTGAT,Srsf9,Cited1,0.5024
4,Ductal,AAACCTGAGCCTTGAT,Med13,Prdm16,0.4914
5,Ngn3 low EP,AAACGGGTCAGCTCTC,Onecut1,Ddx5,0.4873
6,Ngn3 low EP,AAACGGGTCAGCTCTC,Onecut1,Edf1,0.4826
7,Ngn3 low EP,AAACGGGTCAGCTCTC,Onecut1,Tead2,0.4779
8,Ngn3 low EP,AAACGGGTCAGCTCTC,Onecut1,Nkx6-1,0.4778
9,Ngn3 low EP,AAACGGGTCAGCTCTC,Onecut1,Snrpb,0.4731


## 6. Optional: save all cell-specific GRNs

The cell below is disabled by default. When enabled, it streams the selected edges for all cells to one CSV without storing every reconstructed network in memory.

The output columns are:

```text
cell_id,Gene1,Gene2,score
```

`EDGE_SELECTION` controls the exported network:

- `"threshold"`: retain edges with `score >= 0.1`, matching the main tutorial;
- `"top_k"`: retain the 1000 highest-scoring edges independently in every cell.

Threshold and top-k filtering are mutually exclusive. You can adjust `SCORE_THRESHOLD` or `TOP_K_EDGES` below. Leaving both filters as `None` exports every raw edge and produces an extremely large file.

Writing either selection iterates the lazy result and runs inference for all cells. Existing files are not overwritten automatically.


In [6]:
SAVE_CELL_SPECIFIC_GRNS = False
EDGE_SELECTION = "threshold"  # Choose "threshold" or "top_k".
TOP_K_EDGES = 1000

if SAVE_CELL_SPECIFIC_GRNS:
    if EDGE_SELECTION not in {"threshold", "top_k"}:
        raise ValueError(
            "EDGE_SELECTION must be 'threshold' or 'top_k'."
        )

    export_grns = inferencer.infer_cell_specific(
        adata,
        batch_size=8,
        score_threshold=(
            SCORE_THRESHOLD if EDGE_SELECTION == "threshold" else None
        ),
        top_k_edges=(
            TOP_K_EDGES if EDGE_SELECTION == "top_k" else None
        ),
    )
    selection_label = (
        f"threshold{SCORE_THRESHOLD}"
        if EDGE_SELECTION == "threshold"
        else f"top{TOP_K_EDGES}"
    )

    output_dir = REPO_ROOT / "results" / "cell_specific_grns"
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = (
        output_dir
        / f"mPancreas_cell_specific_grns_{selection_label}.csv"
    )
    write_cell_specific_grns_csv(
        export_grns,
        output_path,
        overwrite=False,
    )
    print(f"Saved cell-specific GRNs to {output_path}")
else:
    print(
        "CSV export is disabled. Set "
        "SAVE_CELL_SPECIFIC_GRNS = True to save all cells."
    )


CSV export is disabled. Set SAVE_CELL_SPECIFIC_GRNS = True to save all cells.


## What you generated

`cell_grns` represents one thresholded TF-by-gene GRN for every retained cell. Its edge tables contain only relationships with `score >= 0.1`. You can iterate over score batches for further analysis, inspect cell-level edge tables, or use the optional export cell to save thresholded or per-cell top-1000 networks.
